# Export the final model's detections for the VALIDATION split

This notebook runs the chosen model on the **validation** images only and saves every detection (down to confidence 0.001) to a JSON file on your Drive. Scoring, threshold choice and confidence intervals are then done on your PC with the torch-free code in this repo.

**The test split is never predicted here.** It is predicted exactly once, at the very end, by a separate step that refuses to run twice.

Before you start: Runtime > Change runtime type > **T4 GPU**. `clean.zip` and `code.zip` must be in `MyDrive/vehicle-detection/`, and the run folder `runs/<RUN_NAME>/weights/best.pt` must exist.

In [ ]:
# ---- EDIT ONLY THIS CELL ----
RUN_NAME = "v8n_seed0"  # the chosen final model
SPLIT = "val"
IMGSZ = 672
CONF = 0.001  # keep very low: the real threshold is chosen later, when scoring
IOU = 0.7  # NMS IoU, the Ultralytics default
CODE_SOURCE = "drive_zip"  # "drive_zip" while the repo is private, "github" once it is public
GITHUB_REPO = "https://github.com/zohabaamir-ai/vehicle-detection-yolov8.git"
DRIVE_ROOT = "/content/drive/MyDrive/vehicle-detection"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip install -q ultralytics==8.4.174

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile

CODE_DIR = "/content/code"
if CODE_SOURCE == "drive_zip":
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    with zipfile.ZipFile(f"{DRIVE_ROOT}/code.zip") as z:
        z.extractall(CODE_DIR)
else:
    if not os.path.exists(CODE_DIR):
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, CODE_DIR], check=True)
for name in [m for m in sys.modules if m.startswith("vehicle_detection")]:
    del sys.modules[name]  # forget any older copy loaded earlier in this session
sys.path.insert(0, f"{CODE_DIR}/src")  # our package needs only numpy, pillow, pyyaml (in Colab)

import vehicle_detection

print("vehicle_detection", vehicle_detection.__version__, "loaded from", vehicle_detection.__file__)

In [ ]:
DATA_DIR = "/content/data/clean"  # keep this path: a resumed run remembers it
from vehicle_detection.verify import verify_dataset


def unpack_dataset():
    shutil.rmtree("/content/data", ignore_errors=True)
    os.makedirs("/content/data", exist_ok=True)
    with zipfile.ZipFile(f"{DRIVE_ROOT}/clean.zip") as z:
        z.extractall("/content/data")


def check_dataset():
    return verify_dataset(
        DATA_DIR, f"{CODE_DIR}/results/split_manifest.csv", f"{CODE_DIR}/results/split_summary.json"
    )


if not os.path.exists(DATA_DIR):
    unpack_dataset()
report = check_dataset()
if not report["ok"]:  # an older copy from earlier in this session: unpack the Drive copy again
    print("Dataset on this machine does not match; unpacking clean.zip again")
    unpack_dataset()
    report = check_dataset()
print(report)
assert report["ok"], "The dataset on Drive does not match the committed split manifest"

In [ ]:
assert SPLIT == "val", "The test split is predicted once, at the very end, not in this notebook"
from vehicle_detection.predict import predict_split

meta = predict_split(
    weights=f"{DRIVE_ROOT}/runs/{RUN_NAME}/weights/best.pt",
    images_dir=f"{DATA_DIR}/{SPLIT}/images",
    out_json=f"{DRIVE_ROOT}/predictions/{RUN_NAME}_{SPLIT}.json",
    conf=CONF,
    iou=IOU,
    imgsz=IMGSZ,
)
print(meta)

## Next

Download `predictions/<RUN_NAME>_val.json` from your Drive to your PC (folder `vehicle-detection-data\\predictions`) and run the evaluation command from the chat.